# DeepWeeds Lab Day 2 — Trần Thị Như Ý

MSHV: **2A202602372**. Notebook workflow for fold 0. The Python modules in `starter/` implement loading, training, inference, benchmarking, and workbook generation. Run model experiments on a GPU runtime, choose configurations only on validation, and evaluate the locked final configuration on test once per seed.


## 0. Install and connect the repository

In [ ]:
!pip -q install timm openpyxl scikit-learn matplotlib
import platform
import sys
from pathlib import Path

REPO_DIR = Path('/content/K4-Track4-Day2-Deeplearning-Advance')
if not REPO_DIR.exists():
    !git clone https://github.com/nhuY02/K4-Track4-Day2-Deeplearning-Advance.git {REPO_DIR}
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'starter'))
import timm
import torch

print('Python', platform.python_version(), '| torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')


## 1. Download and validate fold 0 data

The image archive is large. Keep it on the notebook's persistent storage if available. Fold CSV files are downloaded unchanged from the dataset authors. The archive contains image files at its root, so the image directory below is `data`.


In [ ]:
import hashlib
import urllib.request
import zipfile

DATA_DIR = REPO_DIR / 'data'
LABELS_DIR = DATA_DIR / 'labels'
DATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / 'images.zip'
if not archive.exists():
    urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', archive)
md5 = hashlib.md5(archive.read_bytes()).hexdigest()
assert md5 == 'b7b30f96d466fba86016aa5a26606e0f', f'Image archive MD5 mismatch: {md5}'
with zipfile.ZipFile(archive) as bundle:
    image_count = sum(name.lower().endswith('.jpg') for name in bundle.namelist())
base = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/'
for filename in ('labels.csv', 'train_subset0.csv', 'val_subset0.csv', 'test_subset0.csv'):
    urllib.request.urlretrieve(base + filename, LABELS_DIR / filename)
IMAGES_DIR = archive
print('Validated archive image entries:', image_count, '| labels:', sorted(p.name for p in LABELS_DIR.glob('*.csv')))


## 2. EDA and split validation

In [ ]:
import dataset
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
split_report = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
print(pd.DataFrame(split_report['per_class']).fillna(0).astype(int))
counts = pd.DataFrame(split_report['per_class']).fillna(0).T
counts.plot(kind='bar', figsize=(14, 5), stacked=True)
plt.ylabel('Image count'); plt.tight_layout(); plt.show()
def open_sample_image(filename):
    with zipfile.ZipFile(IMAGES_DIR) as bundle, bundle.open(filename) as stream:
        return Image.open(stream).convert('RGB').copy()
fig, axes = plt.subplots(9, 3, figsize=(8, 20))
for label in range(9):
    examples = train_df[train_df.Label == label].head(3)
    for column, (_, row) in enumerate(examples.iterrows()):
        axes[label, column].imshow(open_sample_image(row.Filename))
        axes[label, column].axis('off')
        axes[label, column].set_title(dataset.CLASS_NAMES[label])
plt.tight_layout(); plt.show()


## 3. Pipeline checks

In [ ]:
import math

import losses
import model as model_utils
import torch
import train
from torch.utils.data import DataLoader

train.set_seed(0)
network = model_utils.build_model('resnet50', pretrained=False, init='scratch').to('cuda' if torch.cuda.is_available() else 'cpu')
probe = torch.randn(8, 3, 224, 224, device=next(network.parameters()).device)
labels = torch.randint(0, 9, (8,), device=probe.device)
initial_loss = torch.nn.functional.cross_entropy(network(probe), labels).item()
print('Initial CE:', initial_loss, '| uniform reference:', math.log(9))
criterion = losses.build_criterion('focal', gamma=0)
assert torch.allclose(criterion(network(probe), labels), torch.nn.functional.cross_entropy(network(probe), labels), atol=1e-6)
small_df = train_df.head(8)
small_ds = dataset.DeepWeedsDataset(small_df, IMAGES_DIR, dataset.build_transforms(False, 224))
small_loader = DataLoader(small_ds, batch_size=8)
print('Pipeline batch:', next(iter(small_loader))[0].shape, '| focal gamma=0 matches CE')


## 4. Backbone comparison

Runs use the common baseline recipe, one seed, and validation checkpoint selection. The list covers ResNet, ConvNeXt, a transformer, and a lightweight network; add more backbones to reach at least five. Training all entries requires a GPU runtime and several hours.


In [ ]:
from train import Config, run

backbones = ['resnet50', 'resnext50_32x4d', 'convnext_tiny', 'deit_small_patch16_224', 'efficientnet_b0']
backbone_results = []
for index, backbone in enumerate(backbones, start=1):
    backbone_results.append(run(Config(exp_id=f'B{index:02d}', backbone=backbone, seed=0,
                                       images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR))))
pd.DataFrame(backbone_results)


## 5. Controlled training ablations

In [ ]:
base = Config(exp_id='T00', backbone='resnet50', seed=0, images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR))
ablations = [
    ('T01', {'init': 'scratch'}), ('T02', {'init': 'frozen'}),
    ('T03', {'aug': 'color'}), ('T04', {'mix': 'cutmix'}),
    ('T05', {'loss': 'ls', 'label_smoothing': 0.1}), ('T06', {'loss': 'focal', 'focal_gamma': 2.0}),
]
training_results = [run(base)]
for exp_id, changes in ablations:
    training_results.append(run(__import__('dataclasses').replace(base, exp_id=exp_id, **changes)))
pd.DataFrame(training_results)


## 6. Validation-only inference study and latency

Use the saved validation logits and `starter/inference.py` to compare horizontal-flip TTA, multi-crop or multi-scale views, probability versus logit aggregation, ensembling, and temperature scaling. Fit temperature on validation only. Measure candidate methods on the target hardware with `starter/benchmark.py`; record GPU, dtype, batch size, resolution, and preprocessing policy.


In [ ]:
import benchmark
import inference

# Example API checks using held-out validation logits from a completed run.
logits_path = Path('runs/B01/seed0/val_logits.npz')
if logits_path.exists():
    saved = __import__('numpy').load(logits_path)
    temperature = inference.fit_temperature(saved['logits'], saved['y_true'])
    calibrated_probs = inference.apply_temperature(saved['logits'], temperature)
    print('Validation-fitted temperature:', temperature, '| probability rows:', calibrated_probs.shape[0])
print('Latency API available:', benchmark.latency_report.__name__)


## 7. Final models, predictions, and grading

In [ ]:
# After selecting the final recipe using validation, run each final and baseline seed once.
final_runs = []
for seed in (0, 1, 2):
    final_runs.append(run(Config(exp_id='F01', backbone='resnet50', seed=seed,
        images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR), save_test_predictions=True)))
    final_runs.append(run(Config(exp_id='T00', backbone='resnet50', seed=seed,
        images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR), save_test_predictions=True)))


In [ ]:
!python {REPO_DIR}/eval.py score --pred 'predictions/F01_seed*_test.csv' --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred 'predictions/T00_seed*_test.csv' --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out
!python {REPO_DIR}/eval.py grade --final 'predictions/F01_seed*_test.csv' --baseline 'predictions/T00_seed*_test.csv' --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out


## 8. Export workbook and complete the written analysis

In [ ]:
!python {REPO_DIR}/starter/reporting.py --runs runs --output results.xlsx


The workbook is generated from recorded runs; add inference and final-evaluation records to their named sheets. Complete `report.md` from the real workbook, include confusion-matrix and per-class error analysis, and record notebook/runtime versions. Never enter unmeasured metrics as experimental results.
